<a href="https://colab.research.google.com/github/shubhishukla8176-arch/exfoliation-energy-ml/blob/main/notebooks/03_composition_descriptors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip -q install matminer pymatgen

print("Matminer and Pymatgen installed successfully!")


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Matminer and Pymatgen installed successfully!


In [2]:

import os
import json
import numpy as np
import pandas as pd

from collections import Counter
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty, Stoichiometry

print("All libraries imported successfully!")


All libraries imported successfully!


In [3]:

from google.colab import drive

drive.mount('/content/drive')

# Location of the project data
project_folder = "/content/drive/MyDrive/MASC520_Project"

# Load the existing dataset
json_path = os.path.join(
    project_folder,
    "exfoliation_energy_dataset.json"
)

with open(json_path, "r") as file:
    records = json.load(file)

print("Dataset loaded successfully!")
print("Total materials:", len(records))

print("\nFirst material ID:", records[0]["jid"])
print("Elements in first material:", records[0]["atoms"]["elements"])


Mounted at /content/drive
Dataset loaded successfully!
Total materials: 813

First material ID: JVASP-10
Elements in first material: ['V', 'Se', 'Se']


In [4]:

composition_objects = []
composition_information = []

for record in records:

    jid = record["jid"]

    # Extract the elements from the crystal structure
    elements = record["atoms"]["elements"]

    # Count the number of atoms of each element
    element_counts = Counter(elements)

    # Create a Pymatgen composition
    comp = Composition(dict(element_counts))

    composition_objects.append(comp)

    composition_information.append({
        "jid": jid,
        "formula": comp.reduced_formula,
        "element_counts": dict(element_counts),
        "element_fractions": comp.fractional_composition.get_el_amt_dict()
    })

composition_df = pd.DataFrame(composition_information)

print("Composition calculation completed!")
display(composition_df.head())


Composition calculation completed!


,jid,formula,element_counts,element_fractions
0,JVASP-10,VSe2,"{'V': 1, 'Se': 2}","{'V': 0.3333333333333333, 'Se': 0.666666666666..."
1,JVASP-60702,Pd(Se4Cl)2,"{'Pd': 4, 'Se': 32, 'Cl': 8}","{'Pd': 0.09090909090909091, 'Se': 0.7272727272..."
2,JVASP-28397,SiS,"{'Si': 4, 'S': 4}","{'Si': 0.5, 'S': 0.5}"
3,JVASP-3534,BiClO,"{'Bi': 2, 'Cl': 2, 'O': 2}","{'Bi': 0.3333333333333333, 'Cl': 0.33333333333..."
4,JVASP-323,TiS2,"{'Ti': 1, 'S': 2}","{'Ti': 0.3333333333333333, 'S': 0.666666666666..."


In [5]:

# Identify all unique elements in the dataset

all_elements = sorted({
    element
    for item in composition_information
    for element in item["element_fractions"].keys()
})

print("Number of unique elements:", len(all_elements))
print("Elements:", all_elements)

# Create one fraction column for every element

fraction_rows = []

for item in composition_information:

    fractions = item["element_fractions"]

    row = {"jid": item["jid"]}

    for element in all_elements:
        row[f"fraction_{element}"] = fractions.get(element, 0.0)

    fraction_rows.append(row)

fraction_df = pd.DataFrame(fraction_rows)

print("\nElement fraction features:")
display(fraction_df.head())


Number of unique elements: 78
Elements: ['Ag', 'Al', 'As', 'Au', 'B', 'Ba', 'Be', 'Bi', 'Br', 'C', 'Ca', 'Cd', 'Ce', 'Cl', 'Co', 'Cr', 'Cu', 'Dy', 'Er', 'F', 'Fe', 'Ga', 'Ge', 'H', 'Hf', 'Hg', 'Ho', 'I', 'In', 'Ir', 'K', 'Kr', 'Li', 'Lu', 'Mg', 'Mn', 'Mo', 'N', 'Na', 'Nb', 'Nd', 'Ni', 'Np', 'O', 'Os', 'P', 'Pb', 'Pd', 'Pr', 'Pt', 'Pu', 'Rb', 'Re', 'Rh', 'Ru', 'S', 'Sb', 'Sc', 'Se', 'Si', 'Sm', 'Sn', 'Sr', 'Ta', 'Tb', 'Tc', 'Te', 'Th', 'Ti', 'Tl', 'Tm', 'U', 'V', 'W', 'Y', 'Yb', 'Zn', 'Zr']

Element fraction features:


,jid,fraction_Ag,fraction_Al,fraction_As,fraction_Au,fraction_B,fraction_Ba,fraction_Be,fraction_Bi,fraction_Br,...,fraction_Ti,fraction_Tl,fraction_Tm,fraction_U,fraction_V,fraction_W,fraction_Y,fraction_Yb,fraction_Zn,fraction_Zr
0,JVASP-10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.000000,0.0,0.0,0.0,0.333333,0.0,0.0,0.0,0.0,0.0
1,JVASP-60702,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
2,JVASP-28397,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
3,JVASP-3534,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.333333,0.0,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
4,JVASP-323,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.333333,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0


In [6]:

fraction_columns = [
    col for col in fraction_df.columns
    if col.startswith("fraction_")
]

fraction_sums = fraction_df[fraction_columns].sum(axis=1)

print("Minimum fraction sum:", fraction_sums.min())
print("Maximum fraction sum:", fraction_sums.max())

print(
    "All fractions sum to 1:",
    np.allclose(fraction_sums, 1.0)
)


Minimum fraction sum: 0.9999999999999998
Maximum fraction sum: 1.0
All fractions sum to 1: True


In [7]:

# Initialize the Magpie composition featurizer

magpie = ElementProperty.from_preset("magpie")

# Obtain descriptor names
magpie_labels = magpie.feature_labels()

print("Number of Magpie descriptors:", len(magpie_labels))

print("\nSome descriptor names:")
print(magpie_labels[:15])


Number of Magpie descriptors: 132

Some descriptor names:
['MagpieData minimum Number', 'MagpieData maximum Number', 'MagpieData range Number', 'MagpieData mean Number', 'MagpieData avg_dev Number', 'MagpieData mode Number', 'MagpieData minimum MendeleevNumber', 'MagpieData maximum MendeleevNumber', 'MagpieData range MendeleevNumber', 'MagpieData mean MendeleevNumber', 'MagpieData avg_dev MendeleevNumber', 'MagpieData mode MendeleevNumber', 'MagpieData minimum AtomicWeight', 'MagpieData maximum AtomicWeight', 'MagpieData range AtomicWeight']


In [8]:

magpie_features = []

for i, comp in enumerate(composition_objects):

    values = magpie.featurize(comp)

    magpie_features.append(values)

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1} materials")

magpie_df = pd.DataFrame(
    magpie_features,
    columns=[f"magpie_{label}" for label in magpie_labels]
)

magpie_df.insert(
    0,
    "jid",
    [record["jid"] for record in records]
)

print("\nMagpie descriptor calculation completed!")
print("Descriptor table shape:", magpie_df.shape)

display(magpie_df.head())


Processed 100 materials
Processed 200 materials
Processed 300 materials
Processed 400 materials
Processed 500 materials
Processed 600 materials
Processed 700 materials
Processed 800 materials

Magpie descriptor calculation completed!
Descriptor table shape: (813, 133)


,jid,magpie_MagpieData minimum Number,magpie_MagpieData maximum Number,magpie_MagpieData range Number,magpie_MagpieData mean Number,magpie_MagpieData avg_dev Number,magpie_MagpieData mode Number,magpie_MagpieData minimum MendeleevNumber,magpie_MagpieData maximum MendeleevNumber,magpie_MagpieData range MendeleevNumber,...,magpie_MagpieData range GSmagmom,magpie_MagpieData mean GSmagmom,magpie_MagpieData avg_dev GSmagmom,magpie_MagpieData mode GSmagmom,magpie_MagpieData minimum SpaceGroupNumber,magpie_MagpieData maximum SpaceGroupNumber,magpie_MagpieData range SpaceGroupNumber,magpie_MagpieData mean SpaceGroupNumber,magpie_MagpieData avg_dev SpaceGroupNumber,magpie_MagpieData mode SpaceGroupNumber
0,JVASP-10,23.0,34.0,11.0,30.333333,4.888889,34.0,46.0,89.0,43.0,...,0.000000,0.000000,0.00000,0.0,14.0,229.0,215.0,85.666667,95.555556,14.0
1,JVASP-60702,17.0,46.0,29.0,32.000000,5.454545,34.0,62.0,94.0,32.0,...,0.000000,0.000000,0.00000,0.0,14.0,225.0,211.0,42.272727,41.123967,14.0
2,JVASP-28397,14.0,16.0,2.0,15.000000,1.000000,14.0,78.0,88.0,10.0,...,0.000000,0.000000,0.00000,0.0,70.0,227.0,157.0,148.500000,78.500000,70.0
3,JVASP-3534,8.0,83.0,75.0,36.000000,31.333333,8.0,86.0,94.0,8.0,...,0.000000,0.000000,0.00000,0.0,12.0,64.0,52.0,29.333333,23.111111,12.0
4,JVASP-323,16.0,22.0,6.0,18.000000,2.666667,16.0,43.0,88.0,45.0,...,0.000023,0.000008,0.00001,0.0,70.0,194.0,124.0,111.333333,55.111111,70.0


In [9]:

# Initialize stoichiometry featurizer

stoich = Stoichiometry()

stoich_labels = stoich.feature_labels()

print("Stoichiometry descriptor names:")
print(stoich_labels)

# Calculate descriptors

stoich_features = []

for comp in composition_objects:
    values = stoich.featurize(comp)
    stoich_features.append(values)

stoich_df = pd.DataFrame(
    stoich_features,
    columns=[f"stoich_{label}" for label in stoich_labels]
)

stoich_df.insert(
    0,
    "jid",
    [record["jid"] for record in records]
)

print("\nStoichiometry features calculated!")
display(stoich_df.head())


Stoichiometry descriptor names:
['0-norm', '2-norm', '3-norm', '5-norm', '7-norm', '10-norm']

Stoichiometry features calculated!


,jid,stoich_0-norm,stoich_2-norm,stoich_3-norm,stoich_5-norm,stoich_7-norm,stoich_10-norm
0,JVASP-10,2,0.745356,0.693361,0.670782,0.667408,0.666732
1,JVASP-60702,3,0.755148,0.731509,0.727419,0.727279,0.727273
2,JVASP-28397,2,0.707107,0.629961,0.574349,0.552045,0.535887
3,JVASP-3534,3,0.577350,0.480750,0.415244,0.389977,0.372041
4,JVASP-323,2,0.745356,0.693361,0.670782,0.667408,0.666732


In [10]:

# Start with element fractions
composition_features = fraction_df.copy()

# Add Magpie descriptors
composition_features = composition_features.merge(
    magpie_df,
    on="jid"
)

# Add stoichiometry descriptors
composition_features = composition_features.merge(
    stoich_df,
    on="jid"
)

# Add target property
target_df = pd.DataFrame({
    "jid": [record["jid"] for record in records],
    "exfoliation_energy": [
        record["exfoliation_energy"] for record in records
    ]
})

composition_features = composition_features.merge(
    target_df,
    on="jid"
)

print("Final composition dataset created!")

print("Number of materials:", len(composition_features))
print("Number of columns:", len(composition_features.columns))

display(composition_features.head())


Final composition dataset created!
Number of materials: 813
Number of columns: 218


,jid,fraction_Ag,fraction_Al,fraction_As,fraction_Au,fraction_B,fraction_Ba,fraction_Be,fraction_Bi,fraction_Br,...,magpie_MagpieData mean SpaceGroupNumber,magpie_MagpieData avg_dev SpaceGroupNumber,magpie_MagpieData mode SpaceGroupNumber,stoich_0-norm,stoich_2-norm,stoich_3-norm,stoich_5-norm,stoich_7-norm,stoich_10-norm,exfoliation_energy
0,JVASP-10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,85.666667,95.555556,14.0,2,0.745356,0.693361,0.670782,0.667408,0.666732,83.78
1,JVASP-60702,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,42.272727,41.123967,14.0,3,0.755148,0.731509,0.727419,0.727279,0.727273,88.81
2,JVASP-28397,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,148.500000,78.500000,70.0,2,0.707107,0.629961,0.574349,0.552045,0.535887,0.03
3,JVASP-3534,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.333333,0.0,...,29.333333,23.111111,12.0,3,0.577350,0.480750,0.415244,0.389977,0.372041,71.05
4,JVASP-323,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,111.333333,55.111111,70.0,2,0.745356,0.693361,0.670782,0.667408,0.666732,89.57


In [11]:

# Check missing and infinite values in numerical descriptors

numeric_features = composition_features.select_dtypes(
    include=[np.number]
)

print("Missing values:", numeric_features.isna().sum().sum())

print(
    "Infinite values:",
    np.isinf(numeric_features.to_numpy()).sum()
)

# Save the composition feature dataset

composition_folder = os.path.join(
    project_folder,
    "composition_features"
)

os.makedirs(composition_folder, exist_ok=True)

output_path = os.path.join(
    composition_folder,
    "composition_descriptors.csv"
)

composition_features.to_csv(output_path, index=False)

print("\nComposition descriptors saved!")
print(output_path)


Missing values: 0
Infinite values: 0

Composition descriptors saved!
/content/drive/MyDrive/MASC520_Project/composition_features/composition_descriptors.csv
